# 01 — KV Cache and Generation (A-Day 3 + A-Day 6)

1. Add KV cache to the generation loop (from scratch) and measure speedup.
2. Use vLLM to measure Prefill vs Decode (TTFT, ITL).
3. KV cache memory formula and prefix caching.
4. MQA / GQA / MLA memory savings at 128K context.

## 1. KV cache from scratch (A-Day 3)

Use the same model as NB0. Implement a generate loop that passes `past_key_values` to `model.forward()` and reuses the returned cache so we only compute K,V for the new token.

In [ ]:
import torch
import torch.nn.functional as F
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()
eos_id = tokenizer.eos_token_id

In [ ]:
def generate_with_kv_cache(model, tokenizer, prompt, max_new_tokens=50):
    """Autoregressive loop WITH past_key_values: only one new token per step."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]  # (1, seq_len)
    generated = list(input_ids[0].tolist())
    past_key_values = None

    with torch.no_grad():
        for _ in range(max_new_tokens):
            if past_key_values is None:
                # Prefill: full prompt
                outputs = model(input_ids=input_ids, use_cache=True)
            else:
                # Decode: only the last token
                last_token = torch.tensor([[generated[-1]]], device=model.device, dtype=torch.long)
                outputs = model(input_ids=last_token, past_key_values=past_key_values, use_cache=True)
            past_key_values = outputs.past_key_values
            logits = outputs.logits[:, -1, :]
            next_token = logits.argmax(dim=-1).item()
            generated.append(next_token)
            if next_token == eos_id:
                break
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
def generate_no_cache(model, tokenizer, prompt, max_new_tokens=50):
    """Same as NB0: no past_key_values, full sequence every step."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    generated = list(inputs["input_ids"][0].tolist())

    with torch.no_grad():
        for _ in range(max_new_tokens):
            ids = torch.tensor([generated], device=model.device)
            outputs = model(input_ids=ids)
            logits = outputs.logits[:, -1, :]
            next_token = logits.argmax(dim=-1).item()
            generated.append(next_token)
            if next_token == eos_id:
                break
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
prompt = "The capital of France is"
out_no_cache = generate_no_cache(model, tokenizer, prompt, max_new_tokens=20)
out_with_cache = generate_with_kv_cache(model, tokenizer, prompt, max_new_tokens=20)
print("No cache:", out_no_cache)
print("With KV cache:", out_with_cache)
print("Match:", out_no_cache == out_with_cache)

## 2. Speedup measurement (A-Day 3)

Time both methods for 100 tokens; plot wall time per step to see O(N²) vs O(N) shape.

In [ ]:
def time_steps_no_cache(model, tokenizer, prompt, num_steps=30):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    generated = list(inputs["input_ids"][0].tolist())
    times = []
    with torch.no_grad():
        for step in range(num_steps):
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            t0 = time.perf_counter()
            ids = torch.tensor([generated], device=model.device)
            out = model(input_ids=ids)
            next_tok = out.logits[:, -1, :].argmax(dim=-1).item()
            generated.append(next_tok)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return times

def time_steps_with_cache(model, tokenizer, prompt, num_steps=30):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    generated = list(input_ids[0].tolist())
    past_kv = None
    times = []
    with torch.no_grad():
        for step in range(num_steps):
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            t0 = time.perf_counter()
            if past_kv is None:
                out = model(input_ids=input_ids, use_cache=True)
            else:
                last = torch.tensor([[generated[-1]]], device=model.device, dtype=torch.long)
                out = model(input_ids=last, past_key_values=past_kv, use_cache=True)
            past_kv = out.past_key_values
            next_tok = out.logits[:, -1, :].argmax(dim=-1).item()
            generated.append(next_tok)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return times

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

prompt_short = "Hello."
n_steps = 25
times_no = time_steps_no_cache(model, tokenizer, prompt_short, num_steps=n_steps)
times_yes = time_steps_with_cache(model, tokenizer, prompt_short, num_steps=n_steps)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(range(n_steps), np.array(times_no) * 1000, label="No cache (full seq)")
plt.plot(range(n_steps), np.array(times_yes) * 1000, label="With KV cache")
plt.xlabel("Decode step")
plt.ylabel("Time (ms)")
plt.legend()
plt.title("Wall time per token")
plt.subplot(1, 2, 2)
plt.bar(["No cache", "With cache"], [sum(times_no), sum(times_yes)])
plt.ylabel("Total time (s)")
plt.title("Total time for {} tokens".format(n_steps))
plt.tight_layout()
plt.show()
print(f"Speedup: {sum(times_no)/sum(times_yes):.2f}x")

## 3. vLLM: Prefill vs Decode timing

Send a 500-token prompt to vLLM; capture TTFT and per-token ITL from `RequestOutput.metrics`.

In [ ]:
from vllm import LLM, SamplingParams
from vllm.outputs import RequestOutput

llm = LLM(model=MODEL_ID, trust_remote_code=True, max_model_len=2048)
sampling = SamplingParams(temperature=0, max_tokens=100)

In [ ]:
# Build a ~500-token prompt (repeat a sentence)
base = "Explain in one sentence what machine learning is. "
long_prompt = (base * 80)[:8000]  # ~500 tokens for Qwen tokenizer
actual_tokens = len(tokenizer.encode(long_prompt))
print(f"Prompt length: {actual_tokens} tokens")

In [ ]:
outputs = llm.generate([long_prompt], sampling_params=sampling)
out = outputs[0]
if hasattr(out, 'metrics') and out.metrics is not None:
    m = out.metrics
    ttft = getattr(m, 'time_to_first_token', None) or getattr(m, 'ttft', None)
    itl = getattr(m, 'inter_token_latency', None) or getattr(m, 'time_per_output_token', None)
    if ttft is not None:
        print(f"TTFT (Time to first token): {ttft:.4f} s")
    if itl is not None:
        print(f"ITL (Inter-token latency): {itl*1000:.2f} ms")
    # Alternative: total time and token count
    print(f"Output tokens: {len(out.outputs[0].token_ids)}")
else:
    print("Metrics:", getattr(out, 'metrics', 'N/A'))
    print("Output tokens:", len(out.outputs[0].token_ids))

## 4. KV cache memory formula

Compute `2 × layers × batch × seq_len × hidden_dim × 2 bytes` for Qwen 1.5B; compare with vLLM's actual cache usage.

In [ ]:
# Qwen2.5-1.5B config (from model config)
from transformers import AutoConfig
config = AutoConfig.from_pretrained(MODEL_ID)
num_layers = config.num_hidden_layers
hidden_size = config.hidden_size
num_kv_heads = getattr(config, 'num_key_value_heads', config.num_attention_heads)
head_dim = hidden_size // config.num_attention_heads
batch_size = 1
seq_len = 512
bytes_per_param = 2  # BF16

# Per layer: K and V each (batch, num_kv_heads, seq_len, head_dim)
kv_per_layer = 2 * batch_size * num_kv_heads * seq_len * head_dim * bytes_per_param
total_kv_bytes = num_layers * kv_per_layer
print(f"Layers={num_layers}, hidden={hidden_size}, kv_heads={num_kv_heads}, head_dim={head_dim}")
print(f"KV cache (formula) for batch=1, seq={seq_len}: {total_kv_bytes/1e6:.2f} MB")

## 5. Prefix caching (APC)

Same long system prompt + different questions; compare TTFT with and without `enable_prefix_caching=True`.

In [ ]:
system_prompt = "You are a helpful assistant. " * 200  # ~200 tokens
questions = ["What is 2+2?", "What is the capital of France?", "Say hello."]
prompts_with_prefix = [system_prompt + q for q in questions]

# Without prefix caching
llm_no_prefix = LLM(model=MODEL_ID, trust_remote_code=True, max_model_len=2048, enable_prefix_caching=False)
import time
t0 = time.perf_counter()
outs_no = llm_no_prefix.generate(prompts_with_prefix, sampling_params=SamplingParams(temperature=0, max_tokens=5))
ttft_no = time.perf_counter() - t0  # approximate: time to first output
print(f"Without prefix caching: first batch done in {ttft_no:.3f} s")

In [ ]:
# With prefix caching
llm_prefix = LLM(model=MODEL_ID, trust_remote_code=True, max_model_len=2048, enable_prefix_caching=True)
t0 = time.perf_counter()
outs_yes = llm_prefix.generate(prompts_with_prefix, sampling_params=SamplingParams(temperature=0, max_tokens=5))
ttft_yes = time.perf_counter() - t0
print(f"With prefix caching: first batch done in {ttft_yes:.3f} s")
print(f"Speedup (approx): {ttft_no/ttft_yes:.2f}x")

## 6. MQA / GQA / MLA memory savings at 128K context (A-Day 6)

Pure math: 32 layers, d=4096, seq_len=131072. Compare MHA, GQA, MQA, MLA KV cache bytes.

In [ ]:
num_layers = 32
hidden = 4096
seq_len = 131072
batch = 1
bytes_per_param = 2

def kv_cache_bytes(num_heads, head_dim, num_layers, seq_len, batch=1, bytes_per_param=2):
    return 2 * num_layers * batch * num_heads * seq_len * head_dim * bytes_per_param

head_dim = hidden // 32  # 32 query heads

mha_heads = 32
gqa_kv_heads = 8
mqa_kv_heads = 1

mha_bytes = kv_cache_bytes(mha_heads, head_dim, num_layers, seq_len)
gqa_bytes = kv_cache_bytes(gqa_kv_heads, head_dim, num_layers, seq_len)
mqa_bytes = kv_cache_bytes(mqa_kv_heads, head_dim, num_layers, seq_len)

# MLA: latent_dim=512, assume we store K,V in latent space per head (simplified)
mla_latent_dim = 512
mla_kv_heads = 8
mla_bytes = 2 * num_layers * batch * mla_kv_heads * seq_len * mla_latent_dim * bytes_per_param

print("KV cache at 128K context (32 layers, d=4096, BF16):")
print(f"  MHA (32 heads):  {mha_bytes/1e9:.2f} GB")
print(f"  GQA (8 kv heads): {gqa_bytes/1e9:.2f} GB  ({mha_bytes/gqa_bytes:.1f}x smaller)")
print(f"  MQA (1 kv head):  {mqa_bytes/1e9:.2f} GB  ({mha_bytes/mqa_bytes:.1f}x smaller)")
print(f"  MLA (latent 512): {mla_bytes/1e9:.2f} GB  (~{mha_bytes/mla_bytes:.1f}x smaller)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ["MHA", "GQA", "MQA", "MLA"]
values = [mha_bytes/1e9, gqa_bytes/1e9, mqa_bytes/1e9, mla_bytes/1e9]
plt.bar(labels, values)
plt.ylabel("KV cache (GB)")
plt.title("128K context, 32 layers, d=4096")
plt.show()

In [ ]:
# Verify with Qwen 1.5B config
config = AutoConfig.from_pretrained(MODEL_ID)
n_layers = config.num_hidden_layers
d = config.hidden_size
n_heads = config.num_attention_heads
n_kv = getattr(config, 'num_key_value_heads', n_heads)
hd = d // n_heads
s = 131072
qwen_mha = kv_cache_bytes(n_heads, hd, n_layers, s)
qwen_gqa = kv_cache_bytes(n_kv, hd, n_layers, s)
print(f"Qwen2.5-1.5B: {n_kv} KV heads -> GQA cache {qwen_gqa/1e9:.2f} GB (vs MHA {qwen_mha/1e9:.2f} GB)")